In [100]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [101]:
pip install webdriver-manager


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [102]:
pip install bs4


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [151]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import random
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
import csv
import requests
import pandas as pd
import re

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


In [171]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Modo headless
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


# CODIGO PARA CREAR WINE_DATA
- Cuidado de momento cada vez que ejecuto aplasta lo de antes, hay que cambiar nombre del csv al final
- A ver si para despues lo cambio para que las filas se agreguen a lo que ya esta o si es mas comodo tener un vsc x tipo de vino. 
- Depende también como proceso los datos, si los junto todo antes en un solo txt o si voy poco a poco

**Nota de sabor no funciona, falta tal vez agregar columnas**

In [174]:
# Leer los enlaces desde un archivo de texto
with open('blancos\Vinos blancos de 9 a 11.txt', 'r') as f:
    urls = f.readlines()  # Lee todos los enlaces en el archivo

# Limitar a los primeros 10 enlaces
urls = urls[:3]

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar una lista para guardar todos los datos
all_wine_data = []

# Recorrer cada URL en la lista (solo los primeros 10)
for url in urls:
    url = url.strip()  # Eliminar cualquier espacio en blanco o salto de línea

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Inicializar datos
        wine_data = {}

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        #grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'


        #Grape
        grapes = soup.find_all("a", class_="anchor_anchor__m8Qi- wineFacts__link--3aTg9")
        grape = [grape.text.strip() for grape in grapes if "grapes" in grape["href"]]
        grape = ', '.join(grape) if grape else 'No disponible'


        # Nombre del vino 
        wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
        if wine_headline:
            # Tomamos todo el texto dentro del bloque, sin intentar separarlo
            name = wine_headline.get_text(strip=True)
        else:
            name = 'No disponible'
        
        # Si el nombre del vino contiene el nombre de la bodega, eliminamos la bodega del nombre
        if winery.lower() in name.lower():
            name = name.replace(winery, '').strip()

        # Año
        button_elements = soup.find_all('button', class_='MuiButtonBase-root')
        year = 'No disponible'

        # Buscar en los botones primero
        for button in button_elements:
            if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
                year = button.get('aria-label').strip()
                break

        # Si no se encuentra en los botones, buscar en el span con la clase 'vivino-mui-14ngluw-componentChildren'
        if year == 'No disponible':
            year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
            if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
                year = year_element.text.strip()

        # Si aún no se ha encontrado, buscar todos los 'span' con la clase 'vintageListRow__year--34Tuc' y tomar el primero
        if year == 'No disponible':
            # Extraemos el contenido HTML completo y buscamos con regex
            html_content = str(soup)
            # Buscamos el primer año de 4 dígitos dentro de un span con la clase específica
            match = re.search(r'<span class="vintageListRow__year--34Tuc"[^>]*>(\d{4})</span>', html_content)
            if match:
                year = match.group(1)  # Tomamos el primer año encontrado

        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        if price_element:
            price = price_element.text.replace('€', '').replace('\xa0', '').strip()
        else:
            # Si no lo encuentra, busca el precio en la segunda clase
            price_element = soup.find(class_='purchaseAvailabilityPPC__amount--2_4GT')
            if price_element:
                # Utilizamos regex para encontrar el precio con la coma
                match = re.search(r'\d{1,3}(?:,\d{3})*(?:\.\d+)?', price_element.text)
                if match:
                    price = match.group(0).replace('\xa0', '').strip()
                else:
                    price = 'No disponible'
            else:
                price = 'No disponible'

        
        # Grados de Alcohol
        alcohol_element = soup.find(class_='wineFacts__wineFacts--2Ih8B')
        # Buscar todos los spans dentro de la tabla
        if alcohol_element:
            spans = alcohol_element.find_all('span')
        # Filtrar los spans que contienen un número seguido de '%' (grado de alcohol)
        alcohol = 'No disponible'
        for span in spans:
                # Usamos una expresión regular para buscar un número seguido de '%'
                match = re.search(r'\d+%', span.text.strip())
                if match:
                    alcohol = match.group(0)  # El valor que coincide con la expresión regular
                    alcohol = alcohol.replace('%', '')
                    break  # Detener la búsqueda cuando encontramos el primer grado de alcohol
        else:
            alcohol = 'No disponible'



        # Notas de sabor
        taste_containers = soup.find_all(class_='slider__viewPort--30MrB')
        taste_notes = []
        for container in taste_containers:
            # Encontrar todos los elementos con la clase 'tasteNote__popularKeywords--1gIa2'
            taste_keywords = container.find_all(class_='tasteNote__popularKeywords--1gIa2')
    
            # Recorrer cada uno de los elementos encontrados y extraer el texto
            for keyword in taste_keywords:
                if keyword.text.strip():  # Solo si no está vacío
                    taste_notes.append(keyword.text.strip())

        # Unir todas las notas en una sola cadena, separada por coma
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]

#Caracteristicas
        #Caracteristicas vinos
        for url in urls:
            url = url.strip()  # Eliminar cualquier espacio extra o salto de línea
            driver.get(url)

        # Mapeo de etiquetas y las clases de las barras de progreso
        labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

        # Diccionario para guardar los resultados
        progress_values = {}

        try:
            # Buscar todas las barras de progreso
            progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
            # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
            if len(progress_elements) == len(labels):
                # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
                for i, element in enumerate(progress_elements):
                    left_value = element.get_attribute('style').split('left: ')[1].split('%')[0] if 'left' in element.get_attribute('style') else None
            
                    if left_value:
                        # Convertir a float y convertirlo en una nota del 1 al 10 (dividiendo entre 10)
                        value = round(float(left_value) / 10, 1)
                        # Asignar la etiqueta correspondiente
                        progress_values[labels[i]] = value
                        print(f"{labels[i]}: {value}")
                    else:
                        print(f"No se encontró el atributo 'left' para {labels[i]}.")
            else:
                print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
        except Exception as e:
            print(f"Error durante la extracción: {e}")

        # Cerrar el navegador
        driver.quit()




        # Guardar los datos de esta URL
        wine_data = {
            'Url': url,
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Contenido de alcohol': alcohol,
            'Notas de sabor': taste_notes,
            'Maridajes':', '.join(pairings),
            
        }
        all_wine_data.append(wine_data)

    except Exception as e:
        print(f"Error al procesar la URL {url}: {e}")

# Guardar los resultados en un archivo CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir los datos a un DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)  # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

print(df.head())


Error al procesar la URL https://www.vivino.com/ES/es/schloss-muhlenhof-spatburgunder-blanc-de-noirs-trocken/w/1420124: HTTPConnectionPool(host='localhost', port=62177): Max retries exceeded with url: /session/e7f977870768c07aacb6177758f97b14/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001D2A878FDD0>: Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión'))
Error al procesar la URL https://www.vivino.com/ES/es/schloss-muhlenhof-spatburgunder-blanc-de-noirs-trocken/w/1420124: HTTPConnectionPool(host='localhost', port=62177): Max retries exceeded with url: /session/e7f977870768c07aacb6177758f97b14/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001D2A6CF7F90>: Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión')

IndexError: list index out of range

In [159]:
# Leer los enlaces desde un archivo de texto
with open('espumosos\def_espumoso200.txt', 'r') as f:
    urls = f.readlines()  # Lee todos los enlaces en el archivo

# Limitar a los primeros 10 enlaces
urls = urls[4:7]

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar una lista para guardar todos los datos
all_wine_data = []

# Recorrer cada URL en la lista (solo los primeros 10)
for url in urls:
    url = url.strip()  # Eliminar cualquier espacio en blanco o salto de línea
try:
    print("Waiting for page to load...")
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".tasteNote__tasteNote--wtLz7"))
    )
    # Scroll down the page to ensure dynamic content is loaded
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)  # Wait for content to load after scroll
    # Take a screenshot for debugging
    driver.save_screenshot('vivino_page.png')
    print("Waiting for taste note cards...")
    # Wait again for taste note cards after the scroll
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".tasteNote__tasteNote--wtLz7"))
    )
    print("Taste note cards found. Extracting data...")
    # Find all the taste note card elements
    cards = driver.find_elements(By.CSS_SELECTOR, ".tasteNote__tasteNote--wtLz7")
    if not cards:
        print("No taste note cards found.")
    # Loop through each card and extract relevant data
    for card in cards:
        try:
            print("Extracting taste note data...")
            # Extract the taste note title (e.g., "manzana verde")
            taste_note = card.find_element(By.CSS_SELECTOR, ".tasteNote__popularKeywords--1gIa2").text
            print(f"Taste Note: {taste_note}")
            # Extract the mentions (e.g., "3 menciones sobre notas de frutos de árbol")
            mentions = card.find_element(By.CSS_SELECTOR, ".tasteNote__mentions--1T_d5").text
            print(f"Mentions: {mentions}")
            # Extract the background color (e.g., rgb(129, 150, 50))
            background_color = card.find_element(By.CSS_SELECTOR, ".tasteNote__iconContainer--3k2Oo").value_of_css_property("background-color")
            print(f"Background Color: {background_color}")
            print("="*40)
        except Exception as e:
            print(f"Error extracting data from card: {e}")
            print(f"Error details: {str(e)}")
except Exception as e:
    print(f"Error during scraping: {e}")
    print(f"Error details: {str(e)}")
# Close the driver
driver.quit()

Waiting for page to load...
Error during scraping: Message: 
Stacktrace:
	GetHandleVerifier [0x00DF0B43+25139]
	(No symbol) [0x00D813F4]
	(No symbol) [0x00C604E3]
	(No symbol) [0x00CA83D7]
	(No symbol) [0x00CA872B]
	(No symbol) [0x00CF1002]
	(No symbol) [0x00CCD014]
	(No symbol) [0x00CEE778]
	(No symbol) [0x00CCCDC6]
	(No symbol) [0x00C9BDE9]
	(No symbol) [0x00C9D124]
	GetHandleVerifier [0x010F4373+3185251]
	GetHandleVerifier [0x0111291A+3309578]
	GetHandleVerifier [0x0110CF42+3286578]
	GetHandleVerifier [0x00E87AE0+643536]
	(No symbol) [0x00D8A20D]
	(No symbol) [0x00D870B8]
	(No symbol) [0x00D87257]
	(No symbol) [0x00D79E00]
	BaseThreadInitThunk [0x75835D49+25]
	RtlInitializeExceptionChain [0x776FCDEB+107]
	RtlGetAppContainerNamedObjectPath [0x776FCD71+561]

Error details: Message: 
Stacktrace:
	GetHandleVerifier [0x00DF0B43+25139]
	(No symbol) [0x00D813F4]
	(No symbol) [0x00C604E3]
	(No symbol) [0x00CA83D7]
	(No symbol) [0x00CA872B]
	(No symbol) [0x00CF1002]
	(No symbol) [0x00CCD014]

# URL uno por uno
Todo ok salvo nota de sabor


In [71]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd

# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/mestres-coquet-gran-reserva-brut-nature/w/4311421"

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Solicitud a la página
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

# Inicializar datos
wine_data = {}

try:
    # Nombre del vino: Ahora extraemos solo el texto del nombre (después de la bodega)
    wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
    if wine_headline:
        # Obtener todo el texto dentro del div, luego buscar solo la parte que es el nombre del vino
        text_parts = wine_headline.get_text(strip=True).split(" ")  # Separar por espacios
        # Todo lo que venga después del primer elemento (que sería la bodega)
        name = " ".join(text_parts[1:])  # Tomamos todo después del primer elemento (la bodega)
    else:
        name = 'No disponible'

    #Año :
    # Buscar todos los botones en la página
    button_elements = soup.find_all('button', class_='MuiButtonBase-root')

    # Inicializar año como 'No disponible'
    year = 'No disponible'

    # Iterar sobre todos los botones y buscar uno que contenga un año (4 dígitos)
    for button in button_elements:
        # Verificamos si el aria-label o el texto del botón contiene un año válido
        if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
            year = button.get('aria-label').strip()
            break  # Si encontramos el año, terminamos el bucle

    if year == 'No disponible':
    # Si no encontramos el año en los botones, intentamos otra estrategia (como se hizo anteriormente)
        year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
        if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
            year = year_element.text.strip()




    # País, región, bodega, tipo de vino, uva
    breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
    country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
    region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
    winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
    wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
    grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'

    # Precio
    price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
    price = price_element.text.replace('€', '').replace('\xa0', '').strip() if price_element else 'No disponible'

    # Valoración
    rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
    rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

    # Notas de sabor
    taste_containers = soup.find_all(class_='tasteNote__textContainer--2xPXc')
    taste_notes = []
    for container in taste_containers:
        taste_keywords = container.find(class_='tasteNote__popularKeywords--1gIa2')
        if taste_keywords:
            taste_notes.append(taste_keywords.text.strip())
    taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

    # Maridajes
    food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
    pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]



    # Guardar datos
    wine_data = {
        'Nombre': name,
        'Año': year,
        'País': country,
        'Región': region,
        'Bodega': winery,
        'Tipo de vino': wine_type,
        'Uva': grape,
        'Precio': price,
        'Valoración': rating,
        'Notas de sabor': taste_notes,
        'Maridajes': ', '.join(pairings),
         }

except Exception as e:
    print(f"Error durante la extracción: {e}")

# Guardar en CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=wine_data.keys())
    writer.writeheader()
    writer.writerow(wine_data)

# Convertir a DataFrame y mostrar los primeros registros
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)   # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

df = pd.DataFrame([wine_data])
print(df.head())



                     Nombre   Año    País Región   Bodega   Tipo de vino  \
0  Gran Reserva Brut Nature  2019  España   Cava  Mestres  Vino espumoso   

      Uva Precio Valoración Notas de sabor  \
0  Mezcla  15.87        3.9  No disponible   

                                                                    Maridajes  
0  Marisco, Aperitivos y tentempiés, Pescado blanco, Aperitivo, Carne adobada  


# OTRO CODIGO DE VICENTE
Texto definitivo para coger las url de archivo txt

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time

# Headers para evitar bloqueos
headers = {
    'User -Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar lista para almacenar todos los datos de vino
all_wine_data = []

# Leer las URLs desde un archivo de texto
with open('def_espumoso200.txt', 'r') as file:
    urls = file.readlines()

# Iterar sobre cada URL
for index, url in enumerate(urls, start=1):
    url = url.strip()  # Eliminar espacios en blanco
    wine_data = {}  # Inicializar datos para cada vino

    print(f"Procesando URL {index}/{len(urls)}: {url}")  # Mostrar el progreso

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Nombre y año
        wine_headline = soup.find(class_='wineHeadline-module__vintage--1UHSo')
        if wine_headline:
            name = wine_headline.find('a').text.strip() if wine_headline.find('a') else 'No disponible'
            year = wine_headline.text.strip().split()[-1]  # Última palabra debería ser el año
        else:
            name = 'No disponible'
            year = 'No disponible'

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'

        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        price = price_element.text.replace('€', '').replace('\xa0', '').strip() if price_element else 'No disponible'

        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

        # Notas de sabor
        taste_containers = soup.find_all(class_='tasteNote__textContainer--2xPXc')
        taste_notes = []
        for container in taste_containers:
            taste_keywords = container.find(class_='tasteNote__popularKeywords--1gIa2')
            if taste_keywords:
                taste_notes.append(taste_keywords.text.strip())
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]

        # Guardar datos
        wine_data = {
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Notas de sabor': taste_notes,
            'Maridajes': ', '.join(pairings),
        }

        all_wine_data.append(wine_data)  # Agregar datos a la lista

    except Exception as e:
        print(f"Error durante la extracción de {url}: {e}")

    # Esperar 2 segundos antes de la siguiente solicitud
    time.sleep(2)

# Guardar todos los datos en un CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir a DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'def_espumoso200.txt'